# STAGE 6 — RECOMMENDATION + GEMINI EXPLANATION ENGINE EXPLORATION

This notebook explores and demonstrates the **Recommendation & Gemini Explanation Engine** (Stage 6) of Spendable.

### Key Architectural Separation:
1. **Part A — Deterministic Recommendation Engine**: Fully rule-based, objective source of truth. Generates prioritized recommendation items using factual conditions from Spendable and Scenario outputs.
2. **Part B — Explanation Context**: Facts-only structured JSON payload (`ExplanationContext`) passed downstream to Gemini.
3. **Part C — Gemini Explanation Layer**: Translates structured context into human-readable text strictly adhering to responsible AI rules (no fake facts, no recalculations, no guaranteed financial advice).
4. **Deterministic Fallback**: Ensures 100% service reliability even if Gemini API key is missing or unavailable.


In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import json

# Setup sys.path to backend
sys.path.insert(0, str(Path.cwd() / "backend"))

from app.engine.schema import SpendableOutput, LiquidityState
from app.engine.calculator import SpendableCalculator
from app.recommendation.engine import RecommendationEngine
from app.scenario.simulator import ScenarioSimulator
from app.scenario.schema import ScenarioInput, ScenarioType
from app.explanation.schema import ExplanationContext, GeminiExplanationResponse
from app.explanation.gemini import ExplanationGenerator
from app.forecasting.models import CashFlowForecastModel


## 1. Base Spendable Output & Deterministic Recommendation Generation

We load a sample snapshot and generate prioritized recommendations.


In [ ]:
data_path = Path.cwd() / "data" / "features"
test_df = pd.read_csv(data_path / "features_test.csv")
train_df = pd.read_csv(data_path / "features_train.csv")

model = CashFlowForecastModel()
model.fit(train_df)

sample_row = test_df.iloc[0].to_dict()
user_id = str(sample_row.get("account_id", "USER_001"))
snapshot_time = str(sample_row.get("snapshot_time", "2026-03-01"))
current_balance = float(sample_row.get("current_balance", 0.0))

calc = SpendableCalculator()
forecast = model.predict_snapshot(sample_row)

sp_out = calc.calculate(
    user_id=user_id, snapshot_time=snapshot_time, current_balance=current_balance,
    features=sample_row, commitments=[], forecast=forecast
)

rec_engine = RecommendationEngine()
recs = rec_engine.generate_recommendations(sp_out)

print("--- Base Spendable Engine Output ---")
print(f"Current Balance: ৳{sp_out.current_balance:,.2f}")
print(f"Spendable Amount: ৳{sp_out.spendable_amount:,.2f}")
print(f"Liquidity State: {sp_out.liquidity_state.value}\n")

print("--- Generated Prioritized Recommendations ---")
for r in recs:
    print(f"[{r.priority.value}] {r.title} ({r.type.value})")
    print(f"   Message: {r.message}")
    print(f"   Reason: {r.reason}\n")


--- Base Spendable Engine Output ---
Current Balance: ৳59,835.05
Spendable Amount: ৳22,552.76
Liquidity State: HEALTHY

--- Generated Prioritized Recommendations ---
[INFO] Adaptive Volatility Buffer Active (INCOME_IRREGULARITY)
   Message: Your recent cash flow has shown variability, so your safety buffer has been expanded.
   Reason: Safety reserve includes an additional ৳5,514.07 buffer due to cash flow volatility.



## 2. Facts-Only ExplanationContext Payload

We construct the structured `ExplanationContext` passed to Gemini. Note that it contains only objective facts.


In [ ]:
expl_gen = ExplanationGenerator(api_key="")
context = expl_gen.build_context(sp_out, recs)

print("--- Structured Facts-Only ExplanationContext ---")
print(json.dumps(context.model_dump(), indent=2))


--- Structured Facts-Only ExplanationContext ---
{
  "user_id": "ACC-0497",
  "snapshot_time": "2026-01-31T08:02:18+00:00",
  "current_balance": 59835.05,
  "spendable_amount": 22552.76,
  "protected_amount": 37282.29,
  "planning_horizon_days": 30,
  "expected_inflow": 79159.12,
  "expected_outflow": 41853.04,
  "upcoming_commitments": 0.0,
  "forecasted_minimum_balance": 34284.5,
  "safety_reserve": 11731.74,
  "liquidity_state": "HEALTHY",
  "recommendations": [
    {
      "type": "INCOME_IRREGULARITY",
      "priority": "INFO",
      "title": "Adaptive Volatility Buffer Active",
      "message": "Your recent cash flow has shown variability, so your safety buffer has been expanded.",
      "supporting_amount": 5514.07,
      "supporting_metric": "volatility_addon",
      "reason": "Safety reserve includes an additional \u09f35,514.07 buffer due to cash flow volatility.",
      "action": "Maintain liquidity buffer to cushion against income fluctuations."
    }
  ],
  "scenario": nul

## 3. Human-Readable Explanation Output (with Fallback)

We invoke the `ExplanationGenerator`. When `GEMINI_API_KEY` is not present, it executes the deterministic fallback engine.


In [ ]:
explanation = expl_gen.explain(context)

print("--- Explanation Response (is_fallback=" + str(explanation.is_fallback) + ") ---")
print(f"SUMMARY: {explanation.summary}\n")
print("WHY:")
for w in explanation.why:
    print(f"  - {w}")
print("\nKEY FACTORS:")
for k in explanation.key_factors:
    print(f"  - {k}")
print("\nRECOMMENDATIONS:")
for r in explanation.recommendations:
    print(f"  - {r}")
print(f"\nDISCLAIMER: {explanation.disclaimer}")


--- Explanation Response (is_fallback=True) ---
SUMMARY: You have ৳22,552.76 safely spendable out of your ৳59,835.05 account balance over the next 30 days.

WHY:
  - Your observed balance is ৳59,835.05, but ৳37,282.29 is protected for obligations and safety reserves.
  - Forecasted minimum account balance trajectory is projected at ৳34,284.50.

KEY FACTORS:
  - Adaptive safety reserve buffer: ৳11,731.74
  - Categorical liquidity state: HEALTHY

RECOMMENDATIONS:
  - [INFO] Adaptive Volatility Buffer Active: Your recent cash flow has shown variability, so your safety buffer has been expanded.

DISCLAIMER: This estimate is strictly derived from observed account transaction history up to the snapshot date. It does not include external accounts, physical cash, or unobserved obligations.


## 4. Scenario Simulation + Gemini Explanation Integration

We simulate a hypothetical scenario ("Spend ৳5,000 today") and generate the updated explanation payload.


In [ ]:
sim = ScenarioSimulator(calc)
sc_res = sim.simulate(
    user_id=user_id, snapshot_time=snapshot_time, current_balance=current_balance,
    features=sample_row, commitments=[], forecast=forecast,
    scenario_input=ScenarioInput(scenario_type=ScenarioType.ONE_TIME_EXPENSE, amount=5000.0, description="Spend ৳5,000 today")
)

sc_recs = rec_engine.generate_recommendations(sp_out, sc_res)
sc_context = expl_gen.build_context(sp_out, sc_recs, sc_res)
sc_expl = expl_gen.explain(sc_context)

print("--- Scenario Explanation Output ---")
print(f"SUMMARY: {sc_expl.summary}")
print(f"SCENARIO IMPACT: {sc_expl.scenario_explanation}\n")
print("RECOMMENDATIONS INCLUDING SCENARIO INSIGHT:")
for r in sc_expl.recommendations:
    print(f"  - {r}")


--- Scenario Explanation Output ---
SUMMARY: You have ৳22,552.76 safely spendable out of your ৳59,835.05 account balance over the next 30 days.
SCENARIO IMPACT: Simulating scenario 'Spend ৳5,000 today' results in a decrease of ৳5,000.00 in spendable capacity (from ৳22,552.76 to ৳17,552.76).

RECOMMENDATIONS INCLUDING SCENARIO INSIGHT:
  - [WARNING] Scenario Impact Insight: Simulating 'Spend ৳5,000 today' would reduce your spendable amount by ৳5,000.00.
  - [INFO] Adaptive Volatility Buffer Active: Your recent cash flow has shown variability, so your safety buffer has been expanded.
